#### Identifying the Node with high response rate from above graph and applying OS related Scheduling Algorithm

In [1]:
print("Hello")

Hello


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
mc_rrt = pd.read_csv('MCRRTUpdate_0.csv',on_bad_lines='skip')

In [ ]:
call_graph = pd.read_csv('CallGraph_0.csv',on_bad_lines='skip')

In [4]:
ms_metric = pd.read_csv('MSMetricsUpdate_0.csv',on_bad_lines='skip')

In [3]:
call_sample = pd.read_csv('CallGraph_350k.csv')

In [ ]:
call_graph.head(5)

,timestamp,traceid,service,rpc_id,rpctype,um,uminstanceid,interface,dm,dminstanceid,rt
0,115352,T_11560863075,S_153587416,0.1,rpc,MS_58845,MS_58845_POD_0,xOuy6-80Vt,MS_71712,MS_71712_POD_244,2.0
1,86450,T_22121589080,S_1798353,0.1,rpc,MS_24094,MS_24094_POD_3317,4Db39LxDTC,MS_24094,MS_24094_POD_5168,5.0
2,86450,T_22121589080,S_1798353,0.1,rpc,MS_24094,MS_24094_POD_983,4Db39LxDTC,MS_24094,MS_24094_POD_5168,5.0
3,155753,T_22121586395,S_85905920,0.1,rpc,MS_24094,MS_24094_POD_5463,1oNt-EK5Lm,MS_67455,MS_67455_POD_222,9.0
4,90649,T_22825688569,S_160146479,0.1.1.1,http,MS_29868,MS_29868_POD_22,XQrDXTN_dB,UNKNOWN,UNKNOWN,18.0


In [5]:
ms_metric.head(5)

,timestamp,msname,msinstanceid,nodeid,cpu_utilization,memory_utilization
0,300000,MS_68185,MS_68185_POD_91,NODE_20236,0.172378,0.662361
1,420000,MS_25215,MS_25215_POD_372,NODE_36528,0.085708,0.432862
2,720000,MS_31285,MS_31285_POD_916,NODE_23966,0.445229,0.572093
3,780000,MS_42019,MS_42019_POD_38,NODE_34880,0.091057,0.529339
4,900000,MS_9482,MS_9482_POD_332,NODE_45529,0.177660,0.642626


In [6]:
calling_details = ms_metric.groupby('nodeid').agg(total_calls = ('msname','count')).reset_index()

In [7]:
calling_details.sort_values(by=['total_calls'],ascending=False)

,nodeid,total_calls
1769,NODE_12426,1200
4975,NODE_16745,1140
7970,NODE_20827,1110
13000,NODE_27596,1110
2440,NODE_13337,1080
...,...,...
17469,NODE_33666,4
235,NODE_10311,4
8478,NODE_2150,4
8919,NODE_22097,1


In [8]:
df_joined_ms_metric = pd.merge(
    ms_metric ,
    calling_details,
    on = 'nodeid'
)

In [9]:
df_joined_ms_metric.count()

timestamp             13987704
msname                13987704
msinstanceid          13987704
nodeid                13987704
cpu_utilization       13987704
memory_utilization    13987704
total_calls           13987704
dtype: int64

In [10]:
df_joined_ms_metric.head()

,timestamp,msname,msinstanceid,nodeid,cpu_utilization,memory_utilization,total_calls
0,300000,MS_68185,MS_68185_POD_91,NODE_20236,0.172378,0.662361,660
1,420000,MS_25215,MS_25215_POD_372,NODE_36528,0.085708,0.432862,330
2,720000,MS_31285,MS_31285_POD_916,NODE_23966,0.445229,0.572093,334
3,780000,MS_42019,MS_42019_POD_38,NODE_34880,0.091057,0.529339,900
4,900000,MS_9482,MS_9482_POD_332,NODE_45529,0.177660,0.642626,780


In [11]:
df_joined_ms_metric[['msname','msinstanceid','nodeid']].nunique()

msname           28159
msinstanceid    467613
nodeid           31490
dtype: int64

In [18]:
df_joined_call_graph = pd.merge(
    call_sample ,
    df_joined_ms_metric[['msname','msinstanceid','nodeid','total_calls']],
    left_on = ['um','uminstanceid'],
    right_on = ['msname','msinstanceid']
)

In [19]:
df_joined_call_graph

,timestamp,traceid,service,rpc_id,rpctype,um,uminstanceid,interface,dm,dminstanceid,rt,msname,msinstanceid,nodeid,total_calls
0,99617,T_9007705874,S_67371648,0.4.3.14.5.7,mc,MS_30089,MS_30089_POD_5,7axSflu2ky,MS_72494,MS_72494_POD_72,1.0,MS_30089,MS_30089_POD_5,NODE_14629,600
1,99617,T_9007705874,S_67371648,0.4.3.14.5.7,mc,MS_30089,MS_30089_POD_5,7axSflu2ky,MS_72494,MS_72494_POD_72,1.0,MS_30089,MS_30089_POD_5,NODE_14629,600
2,99617,T_9007705874,S_67371648,0.4.3.14.5.7,mc,MS_30089,MS_30089_POD_5,7axSflu2ky,MS_72494,MS_72494_POD_72,1.0,MS_30089,MS_30089_POD_5,NODE_14629,600
3,99617,T_9007705874,S_67371648,0.4.3.14.5.7,mc,MS_30089,MS_30089_POD_5,7axSflu2ky,MS_72494,MS_72494_POD_72,1.0,MS_30089,MS_30089_POD_5,NODE_14629,600
4,99617,T_9007705874,S_67371648,0.4.3.14.5.7,mc,MS_30089,MS_30089_POD_5,7axSflu2ky,MS_72494,MS_72494_POD_72,1.0,MS_30089,MS_30089_POD_5,NODE_14629,600
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
242288,169643,T_15741397068,S_114078029,0.1.1.2.5,rpc,MS_43813,MS_43813_POD_2,_5Gv8a8pnY,MS_6419,MS_6419_POD_27,33.0,MS_43813,MS_43813_POD_2,NODE_42060,690
242289,169643,T_15741397068,S_114078029,0.1.1.2.5,rpc,MS_43813,MS_43813_POD_2,_5Gv8a8pnY,MS_6419,MS_6419_POD_27,33.0,MS_43813,MS_43813_POD_2,NODE_42060,690
242290,169643,T_15741397068,S_114078029,0.1.1.2.5,rpc,MS_43813,MS_43813_POD_2,_5Gv8a8pnY,MS_6419,MS_6419_POD_27,33.0,MS_43813,MS_43813_POD_2,NODE_42060,690
242291,169643,T_15741397068,S_114078029,0.1.1.2.5,rpc,MS_43813,MS_43813_POD_2,_5Gv8a8pnY,MS_6419,MS_6419_POD_27,33.0,MS_43813,MS_43813_POD_2,NODE_42060,690


In [20]:
df_joined_call_graph.count()

timestamp       242293
traceid         242293
service         242293
rpc_id          242293
rpctype         242293
um              242293
uminstanceid    242293
interface       242293
dm              242293
dminstanceid    242263
rt              242293
msname          242293
msinstanceid    242293
nodeid          242293
total_calls     242293
dtype: int64

##### Building Multiple Logistic Regression Data to Classify the Request as calling Microserice that runs in High contention Node or not

In [110]:
model_data = df_joined_call_graph.copy()

In [111]:
refined_model_data = model_data.drop(columns = ['traceid','service','rt','msname','msinstanceid','timestamp'])

In [112]:
refined_model_data.head(10)

,rpc_id,rpctype,um,uminstanceid,interface,dm,dminstanceid,nodeid,total_calls,is_high_request_node
0,0.4.3.14.5.7,mc,MS_30089,MS_30089_POD_5,7axSflu2ky,MS_72494,MS_72494_POD_72,NODE_14629,600,0
1,0.4.3.14.5.7,mc,MS_30089,MS_30089_POD_5,7axSflu2ky,MS_72494,MS_72494_POD_72,NODE_14629,600,0
2,0.4.3.14.5.7,mc,MS_30089,MS_30089_POD_5,7axSflu2ky,MS_72494,MS_72494_POD_72,NODE_14629,600,0
3,0.4.3.14.5.7,mc,MS_30089,MS_30089_POD_5,7axSflu2ky,MS_72494,MS_72494_POD_72,NODE_14629,600,0
4,0.4.3.14.5.7,mc,MS_30089,MS_30089_POD_5,7axSflu2ky,MS_72494,MS_72494_POD_72,NODE_14629,600,0
5,0.4.3.14.5.7,mc,MS_30089,MS_30089_POD_5,7axSflu2ky,MS_72494,MS_72494_POD_72,NODE_14629,600,0
6,0.4.3.14.5.7,mc,MS_30089,MS_30089_POD_5,7axSflu2ky,MS_72494,MS_72494_POD_72,NODE_14629,600,0
7,0.4.3.14.5.7,mc,MS_30089,MS_30089_POD_5,7axSflu2ky,MS_72494,MS_72494_POD_72,NODE_14629,600,0
8,0.4.3.14.5.7,mc,MS_30089,MS_30089_POD_5,7axSflu2ky,MS_72494,MS_72494_POD_72,NODE_14629,600,0
9,0.4.3.14.5.7,mc,MS_30089,MS_30089_POD_5,7axSflu2ky,MS_72494,MS_72494_POD_72,NODE_14629,600,0


In [113]:
refined_model_data.count()

rpc_id                  242293
rpctype                 242293
um                      242293
uminstanceid            242293
interface               242293
dm                      242293
dminstanceid            242263
nodeid                  242293
total_calls             242293
is_high_request_node    242293
dtype: int64

In [114]:
# refined_model_data.drop_duplicates(inplace = True)

In [115]:
rpc_id_range = dict(zip(refined_model_data['rpc_id'].unique(),range(int(refined_model_data['rpc_id'].nunique()))))
rpc_type_range = dict(zip(refined_model_data['rpctype'].unique(),range(int(refined_model_data['rpctype'].nunique()))))
um_range = dict(zip(refined_model_data['um'].unique(),range(int(refined_model_data['um'].nunique()))))
um_instance_range = dict(zip(refined_model_data['uminstanceid'].unique(),range(int(refined_model_data['uminstanceid'].nunique()))))
interface_range = dict(zip(refined_model_data['interface'].unique(),range(int(refined_model_data['interface'].nunique()))))
dm_range = dict(zip(refined_model_data['dm'].unique(),range(int(refined_model_data['dm'].nunique()))))
dminstanceid_range = dict(zip(refined_model_data['dminstanceid'].unique(),range(int(refined_model_data['dminstanceid'].nunique()))))
nodeid_range = dict(zip(refined_model_data['nodeid'].unique(),range(int(refined_model_data['nodeid'].nunique()))))


In [116]:
refined_model_data['rpc_id'] = refined_model_data['rpc_id'].map(rpc_id_range)
refined_model_data['rpctype'] = refined_model_data['rpctype'].map(rpc_type_range)
refined_model_data['um'] = refined_model_data['um'].map(um_range)
refined_model_data['uminstanceid'] = refined_model_data['uminstanceid'].map(um_instance_range)
refined_model_data['interface'] = refined_model_data['interface'].map(interface_range)
refined_model_data['dm'] = refined_model_data['dm'].map(dm_range)
refined_model_data['dminstanceid'] = refined_model_data['dminstanceid'].map(dminstanceid_range)
refined_model_data['nodeid'] = refined_model_data['nodeid'].map(nodeid_range)

In [117]:
refined_model_data.nunique()

rpc_id                  3228
rpctype                    6
um                      1207
uminstanceid            3234
interface               2469
dm                      1209
dminstanceid            4456
nodeid                  3005
total_calls              270
is_high_request_node       2
dtype: int64

In [118]:
training_data = refined_model_data.dropna()

#### Training and Building the Model

In [119]:
training_data

,rpc_id,rpctype,um,uminstanceid,interface,dm,dminstanceid,nodeid,total_calls,is_high_request_node
0,0,0,0,0,0,0,0.0,0,600,0
1,0,0,0,0,0,0,0.0,0,600,0
2,0,0,0,0,0,0,0.0,0,600,0
3,0,0,0,0,0,0,0.0,0,600,0
4,0,0,0,0,0,0,0.0,0,600,0
...,...,...,...,...,...,...,...,...,...,...
242258,3227,2,93,504,12,12,12.0,498,748,1
242259,3227,2,93,504,12,12,12.0,498,748,1
242260,3227,2,93,504,12,12,12.0,498,748,1
242261,3227,2,93,504,12,12,12.0,498,748,1


In [120]:
training_data.nunique()

rpc_id                  3228
rpctype                    6
um                      1207
uminstanceid            3233
interface               2469
dm                      1209
dminstanceid            4456
nodeid                  3005
total_calls              270
is_high_request_node       2
dtype: int64

In [121]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score

In [ ]:
threshold = training_data['total_calls'].quantile(0.75)
training_data['is_high_request_node'] = (training_data['total_calls'] > threshold).astype(int)

In [123]:
training_data.dropna(inplace = True)

In [124]:
training_data

,rpc_id,rpctype,um,uminstanceid,interface,dm,dminstanceid,nodeid,total_calls,is_high_request_node
0,0,0,0,0,0,0,0.0,0,600,0
1,0,0,0,0,0,0,0.0,0,600,0
2,0,0,0,0,0,0,0.0,0,600,0
3,0,0,0,0,0,0,0.0,0,600,0
4,0,0,0,0,0,0,0.0,0,600,0
...,...,...,...,...,...,...,...,...,...,...
242258,3227,2,93,504,12,12,12.0,498,748,1
242259,3227,2,93,504,12,12,12.0,498,748,1
242260,3227,2,93,504,12,12,12.0,498,748,1
242261,3227,2,93,504,12,12,12.0,498,748,1


In [125]:
X = training_data[[ 'rpctype', 'um', 'uminstanceid', 'interface', 
        'dm', 'dminstanceid', 'nodeid', 'total_calls']]
y = training_data['is_high_request_node']

In [129]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.40, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_model = LogisticRegression(solver='liblinear', max_iter=1000)
log_model.fit(X_train_scaled, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [130]:
y_pred = log_model.predict(X_test_scaled)

# Print results
print("Model Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

importance = pd.DataFrame({
    'Feature': X.columns,
    'Weight': log_model.coef_[0]
}).sort_values(by='Weight', ascending=False)

print("\nFeature Impact on High Request Probability:")
print(importance)

Model Accuracy: 0.9981218913173591

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00     74585
           1       1.00      0.99      1.00     22321

    accuracy                           1.00     96906
   macro avg       1.00      1.00      1.00     96906
weighted avg       1.00      1.00      1.00     96906


Feature Impact on High Request Probability:
        Feature     Weight
7   total_calls  33.474152
6        nodeid   0.668987
3     interface   0.241558
0       rpctype   0.189855
5  dminstanceid   0.019985
1            um  -0.043505
2  uminstanceid  -0.296088
4            dm  -0.385550
